# Supervised Fine-Tuning — Hands-On
Offline numpy notebook for masking, curation, and SFT decisions.

## 0. Setup

In [ ]:
%pip install -q numpy
import numpy as np, hashlib
rng=np.random.RandomState(39)
msgs=[{'role':'user','content':'Define SFT'}, {'role':'assistant','content':'SFT trains on demonstrations.'}]

## 1. Chat template and mask

In [ ]:
def render(messages):
    toks=[]; mask=[]
    for m in messages:
        ts=(f"<{m['role']}> "+m['content']+' <eos>').split()
        toks+=ts; mask += [m['role']=='assistant']*len(ts)
    return toks, np.array(mask)
tokens,mask=render(msgs)
print(list(zip(tokens,mask.astype(int))))

## 2. Masked cross entropy

In [ ]:
V=5; labels=rng.randint(0,V,len(tokens)); raw=rng.normal(size=(len(tokens),V))
p=np.exp(raw-raw.max(1,keepdims=True)); p=p/p.sum(1,keepdims=True); logp=np.log(p)
print('all-token loss', round(float(-logp[np.arange(len(tokens)),labels].mean()),3))
print('assistant loss', round(float(-logp[np.arange(len(tokens))[mask],labels[mask]].mean()),3))

## 3. Deduplicate and split

In [ ]:
rows=['a->b','a->b','rag->fresh facts','json->{ok:true}','bad->']
clean=[r for r in rows if not r.endswith('->')]
uniq=list(dict.fromkeys(clean))
order=sorted(uniq,key=lambda x: hashlib.md5(x.encode()).hexdigest())
train,val=order[:2],order[2:]
print(train,val); assert set(train).isdisjoint(val)

## 4. Forgetting tradeoff

In [ ]:
steps=np.array([0,25,50,100,200])
task=.5+.38*(1-np.exp(-steps/70)); general=.84-.13*(steps/200)**1.2
for s,t,g in zip(steps,task,general): print(s, round(t,3), round(g,3))

## 5. Lever choice

In [ ]:
def lever(fresh, repeated, prompt_fails):
    return 'RAG' if fresh else ('SFT' if repeated and prompt_fails else 'Prompting')
print([lever(*c) for c in [(1,1,1),(0,1,1),(0,0,1)]])

## Exercises
Add policy labels, check validation contamination, and change the masking rule.